In [0]:

dbutils.widgets.text("entity_name", "products")
entity_name = dbutils.widgets.get("entity_name")

In [0]:
%run ../07_Common/00_setup

In [0]:
%run ../07_Common/Utils_Nombre_Archivos_Volumen

In [0]:
%run ../07_Common/01_raw/Utils_Api_Session

In [0]:
%run ../07_Common/01_raw/Utils_Control_Config

In [0]:
%run ../07_Common/01_raw/Utils_Idempotencia_Raw

In [0]:
config = get_raw_config(entity_name)
api_info = get_api_parameters(entity_name)

api_endpoint    = config["api_endpoint"]
raw_path    = config["raw_path"]
data_root_key   = config["data_root_key"]
query_params    = api_info["params"]
offset_param    = api_info["offset_param_name"]
page_size_param = api_info["page_size_param_name"]

execution_date = date.today()
print(f"Procesando entidad: {entity_name} | fecha: {execution_date}")

In [0]:
if file_already_ingested_today(raw_path, entity_name, execution_date):
    print(f"El archivo de '{entity_name}' de hoy ya existe. Se omite la llamada a la API.")
    dbutils.notebook.exit("SKIPPED_ALREADY_INGESTED")

In [0]:

session = get_api_session()
all_records = []
current_skip = int(query_params[offset_param])
page_size = int(query_params[page_size_param])

while True:
    response = session.get(api_endpoint, params=query_params)
    response.raise_for_status()
    data = response.json()

    all_records.extend(data[data_root_key])
    current_skip += page_size
    query_params[offset_param] = str(current_skip)

    if current_skip >= data["total"]:
        break

print(f"Registros extraídos de '{entity_name}': {len(all_records)}")

In [0]:
file_name = build_raw_file_name(entity_name, execution_date)
full_path = f"{raw_path}{file_name}"

contenido_json = json.dumps(all_records, ensure_ascii=False)
dbutils.fs.put(full_path, contenido_json, overwrite=False)

print(f"Archivo guardado en: {full_path}")